# 1. Submission - Logistic Regression

Entrenar el modelo Logit final (9 features significativas) sobre todo el train, predecir sobre test y enviar a Kaggle.

**Modelo:** statsmodels Logit con backward elimination (ROC-AUC CV ~ 0.9485)

**Competencia:** `playground-series-s6e2`

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

## 2. Carga de datos

In [2]:
train = pd.read_csv("../../data/train.csv")
test = pd.read_csv("../../data/test.csv")

# Target binario
train["target"] = (train["Heart Disease"] == "Presence").astype(int)

# Features significativas del modelo final (backward elimination)
features = ["Thallium", "Chest pain type", "Number of vessels fluro",
            "Exercise angina", "Max HR", "ST depression",
            "Slope of ST", "Sex", "EKG results"]

print(f"Train: {train.shape}")
print(f"Test: {test.shape}")
print(f"Features: {len(features)}")

Train: (630000, 16)
Test: (270000, 14)
Features: 9


## 3. Entrenamiento del modelo final

In [3]:
X_train = sm.add_constant(train[features])
y_train = train["target"]

model = sm.Logit(y_train, X_train).fit(disp=0)
print(model.summary())

                           Logit Regression Results                           
Dep. Variable:                 target   No. Observations:               630000
Model:                          Logit   Df Residuals:                   629990
Method:                           MLE   Df Model:                            9
Date:                Fri, 06 Feb 2026   Pseudo R-squ.:                  0.5823
Time:                        18:44:17   Log-Likelihood:            -1.8102e+05
converged:                       True   LL-Null:                   -4.3331e+05
Covariance Type:            nonrobust   LLR p-value:                     0.000
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
const                      -2.6120      0.046    -56.635      0.000      -2.702      -2.522
Thallium                    0.5001      0.002    227.184      0.000       0.496       0.504


## 4. Predicción sobre test

In [4]:
X_test = sm.add_constant(test[features])
test_probs = model.predict(X_test)

# Submission con probabilidades (formato requerido por Kaggle)
submission = pd.DataFrame({
    "id": test["id"],
    "Heart Disease": test_probs.round(4)
})

print(f"Shape: {submission.shape}")
print(f"\nEstadísticas de probabilidades:")
print(submission["Heart Disease"].describe().round(4))
print(f"\nPrimeras filas:")
submission.head(10)

Shape: (270000, 2)

Estadísticas de probabilidades:
count    270000.0000
mean          0.4498
std           0.3995
min           0.0003
25%           0.0501
50%           0.3185
75%           0.9162
max           1.0000
Name: Heart Disease, dtype: float64

Primeras filas:


,id,Heart Disease
0,630000,0.9577
1,630001,0.0032
2,630002,0.9940
3,630003,0.0166
4,630004,0.1649
5,630005,0.9831
6,630006,0.0155
7,630007,0.5664
8,630008,0.9919
9,630009,0.0361


## 5. Guardar CSV

In [5]:
SUBMISSION_FILE = "1_LR_submission.csv"
submission.to_csv(SUBMISSION_FILE, index=False)

# Verificar
check = pd.read_csv(SUBMISSION_FILE)
print(f"Archivo: {SUBMISSION_FILE}")
print(f"Shape: {check.shape}")
print(f"Columnas: {list(check.columns)}")
print(f"IDs: {check['id'].min()} - {check['id'].max()}")
check.head()

Archivo: 1_LR_submission.csv
Shape: (270000, 2)
Columnas: ['id', 'Heart Disease']
IDs: 630000 - 899999


,id,Heart Disease
0,630000,0.9577
1,630001,0.0032
2,630002,0.9940
3,630003,0.0166
4,630004,0.1649


## 6. Submit a Kaggle

In [6]:
COMPETITION = "playground-series-s6e2"
MESSAGE = "LR statsmodels - 9 features significativas"

!kaggle competitions submit -c {COMPETITION} -f {SUBMISSION_FILE} -m "{MESSAGE}"

Successfully submitted to Predicting Heart Disease



  0%|          | 0.00/3.83M [00:00<?, ?B/s]
  0%|          | 16.0k/3.83M [00:00<00:45, 87.7kB/s]
  5%|▍         | 192k/3.83M [00:00<00:04, 785kB/s]  
 13%|█▎        | 512k/3.83M [00:00<00:03, 1.16MB/s]
 62%|██████▏   | 2.36M/3.83M [00:00<00:00, 5.77MB/s]
 80%|████████  | 3.08M/3.83M [00:01<00:00, 2.85MB/s]
100%|██████████| 3.83M/3.83M [00:01<00:00, 2.24MB/s]
